# 19.1 Capstone — CLI Task Tracker

**Prerequisites:** this capstone assumes folders 01–08 and 15, and deliberately ties together
**3.3** comprehensions, **3.4** structural pattern matching, **4.5** type hints,
**5.3** dataclasses & enums, **5.4** dependency injection, **6.2** custom exceptions & chaining,
**07** modules & the standard library, **8.3** JSON, and **15.3–15.5** pytest, fixtures & isolation
**Target:** Python 3.12+

### What you'll build

**`taskr`** — a small command-line task tracker: add tasks with a priority and a due date,
list them (filtered), mark them done, remove them — persisted to a JSON file that survives
between runs. Not a toy snippet and not a framework: one honest, end-to-end program, built
in runnable layers, then **tested with real `pytest` output**.

### How this notebook works

- Every cell runs top-to-bottom with no `input()` and no network.
- Every file `taskr` writes lives in a **throwaway temp directory** — the final cell deletes it.
- Each stage is a working slice: model → persistence → store → CLI → tests.

---

## 1. Problem statement

You want to track work from the terminal. A session should look like this:

```text
$ taskr add 'Ship v2.1 release notes' high 2026-08-28
added #1: Ship v2.1 release notes (high, due 2026-08-28)
$ taskr list --priority high
[ ] #1   high   due 2026-08-28 Ship v2.1 release notes
$ taskr done 1
completed #1: Ship v2.1 release notes
```

### Feature list

| Command | Does |
|---|---|
| `add TITLE [PRIORITY] [DUE]` | create a task; priority defaults to `medium`, due date optional |
| `list [--priority P] [--status S]` | show tasks, optionally filtered |
| `done ID` | mark a task completed |
| `rm ID` | delete a task |

### What "done" looks like

1. Tasks **persist**: a brand-new process pointed at the same file sees the same tasks.
2. Types survive the round trip: a due date goes in as a `date` and **comes back as a `date`**,
   not a string (**8.3**'s lossy-conversion problem, solved on purpose).
3. User mistakes (`done 99`, an unknown command, a bad priority) print a **one-line friendly
   error** — never a traceback. Genuine bugs still crash loudly (**6.2**).
4. The core logic is **testable without touching the real task file** — and we prove it with a
   real `pytest` run at the end (**15.3–15.5**).

## 2. Design decisions — and the *why*

| Decision | Why |
|---|---|
| `@dataclass Task` (**5.3**) | a task is pure data: generated `__init__`/`__repr__`/`__eq__` for free, and `__eq__` is what makes the round-trip *assertable* |
| `StrEnum` for `Priority`/`Status` (**5.3**) | a closed set of values, but each member **is** a `str` — so `"high"` compares equal, JSON serialises it natively, and `Priority("high")` parses it back. A plain string field would accept `"hgih"` silently |
| JSON on disk (**8.3**) | human-readable, standard library, diff-able. ⚠️ JSON is **lossy** — it has no `date` and no enum types — so we convert deliberately: `default=` on the way out, `object_hook` on the way in. Pretending the problem doesn't exist is how `"2026-09-01"`-the-string leaks all over a codebase |
| a `TaskStore` class owning a `pathlib.Path` | *one* place in the whole program touches the filesystem. Everything else calls `store.add(...)`, `store.complete(...)` and never sees a file |
| the path is **injected**, not hard-coded (**5.4**) | `TaskStore(Path.home() / ".taskr.json")` in production, `TaskStore(tmp_path / "tasks.json")` in tests. This one constructor argument is what makes section 5's tests possible — no mocking, no patching, just a different path |
| custom exceptions off one base, `TaskrError` (**6.2**) | the CLI needs exactly **one** `except TaskrError` to turn every *expected* failure into a friendly message, while a genuine bug (`TypeError`, `OSError`) still produces a traceback |
| dispatcher takes **argv-style lists** (**3.4**) | `run_command(store, ["add", "x", "high"])` is callable from a `main()`, from a test, from this notebook — no `input()` loop to fake, no terminal to script |

In [ ]:
# ---- Workspace setup: NOTHING in this notebook touches the repo ----
# Every file taskr writes -- the JSON store, the pytest project at the end --
# lives under this one throwaway directory. The final cell deletes it.
import json
import shlex
import shutil
import subprocess
import sys
import tempfile
import textwrap
from dataclasses import asdict, dataclass
from datetime import date
from enum import StrEnum
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="taskr_capstone_"))

print(f"scratch : {WORK}")
print(f"python  : {sys.version.split()[0]}")

## 3. The build

### Stage 1 — the model

A task is data with no behaviour, which is exactly what `@dataclass` is for (**5.3**). The two
"pick one of a few values" fields become `StrEnum`s: `Priority.HIGH` *is* the string `"high"`,
which pays off twice below — JSON writes it with no converter, and `f"{task.priority}"`
formats it cleanly.

In [ ]:
class Priority(StrEnum):            # (**5.3**) each member IS a str
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


class Status(StrEnum):
    OPEN = "open"
    DONE = "done"


@dataclass
class Task:                         # (**5.3**) __init__/__repr__/__eq__ generated
    id: int
    title: str
    priority: Priority = Priority.MEDIUM
    status: Status = Status.OPEN
    due: date | None = None         # (**4.5**) modern optional syntax


t = Task(id=1, title="Write the capstone", priority=Priority.HIGH,
         due=date(2026, 9, 1))
print(t)

# StrEnum members compare equal to plain strings -- and refuse bad values:
assert t.priority == "high"
assert Priority("high") is Priority.HIGH        # parse a raw string back
print(f"as text: priority={t.priority}, status={t.status}")

### Stage 2 — serialisation, round-tripped and asserted

⚠️ **JSON is lossy** (**8.3**): it has strings, numbers, booleans, `null`, arrays and objects —
and *nothing else*. Two of our fields need explicit help:

| Field | Out (`json.dumps`) | Back (`json.loads`) |
|---|---|---|
| `Priority` / `Status` | free — a `StrEnum` member **is** a `str` | `Priority(raw)` re-validates |
| `date` | ⚠️ `TypeError` without help → `default=` converts to ISO `"2026-09-01"` | `object_hook` calls `date.fromisoformat` |

The test of a serialiser is the **round trip**: `loads(dumps(x)) == x`. Because `@dataclass`
gave `Task` a real `__eq__`, we can assert it.

In [ ]:
def json_default(obj: object) -> str:
    """Called by json.dumps only for types it cannot serialise (**8.3**)."""
    if isinstance(obj, date):
        return obj.isoformat()          # date(2026, 9, 1) -> "2026-09-01"
    raise TypeError(f"not JSON-serialisable: {obj!r}")


TASK_FIELDS = {"id", "title", "priority", "status", "due"}


def task_hook(raw: dict) -> "Task | dict":
    """Rebuild a Task from a plain JSON object (**8.3** object_hook)."""
    if set(raw) != TASK_FIELDS:
        return raw                      # not one of ours -- leave it alone
    return Task(
        id=raw["id"],
        title=raw["title"],
        priority=Priority(raw["priority"]),      # "high" -> Priority.HIGH
        status=Status(raw["status"]),
        due=date.fromisoformat(raw["due"]) if raw["due"] else None,
    )


text = json.dumps([asdict(t)], default=json_default, indent=2)
print(text)

restored = json.loads(text, object_hook=task_hook)
assert restored == [t], "round trip must be lossless"
assert restored[0].due == date(2026, 9, 1)       # a real date, not a string
print(f"\nround trip OK -> {restored[0]}")

### Stage 3 — errors first, then the store

Before writing the store we decide what it *raises*. One base class, `TaskrError`, with the
specific failures underneath (**6.2**):

- the CLI writes **one** `except TaskrError` clause and every expected failure becomes a
  friendly one-liner;
- a genuine bug — `TypeError`, `OSError`, an assertion — is **not** a `TaskrError`, so it still
  crashes with a full traceback, which is what you want during development;
- `TaskNotFoundError` carries the offending id as an **attribute**, not just message text, so
  callers (and tests) can inspect it.

In [ ]:
class TaskrError(Exception):
    """Base for every failure taskr raises deliberately (**6.2**)."""


class TaskNotFoundError(TaskrError):
    """No task exists with the requested id."""

    def __init__(self, task_id: int) -> None:
        super().__init__(f"no task with id {task_id}")
        self.task_id = task_id          # structured data, not just a message


class InvalidCommandError(TaskrError):
    """The command line could not be understood."""


# The hierarchy in action: ONE except clause catches both leaf types.
for exc in (TaskNotFoundError(42), InvalidCommandError("unknown command 'lst'")):
    try:
        raise exc
    except TaskrError as err:
        print(f"caught {type(err).__name__}: {err}")

assert TaskNotFoundError(7).task_id == 7

### Stage 3 (continued) — `TaskStore`, the only code that touches disk

`TaskStore` owns one injected `pathlib.Path` (**5.4**) and hides *all* persistence behind five
methods: `add`, `get`, `list_tasks`, `complete`, `remove`. Each command loads the file, changes
the list, saves the file — the simplest design that makes every run of the CLI see the latest
state. (For a tracker with a few hundred tasks, re-reading a small JSON file per command is
nothing; the extensions in section 6 point at what to do when it isn't.)

⚠️ Note what is *absent*: no hard-coded path, no global state, no `print` — the store returns
values and raises exceptions; **presentation is someone else's job**.

In [ ]:
class TaskStore:
    """All persistence for taskr, behind ONE injected Path (**5.4**, **07**)."""

    def __init__(self, path: Path) -> None:
        self.path = path                # injected: tests pass a tmp_path

    # -- persistence ----------------------------------------------------
    def _load(self) -> list[Task]:
        if not self.path.exists():      # first run: an empty store
            return []
        raw = self.path.read_text(encoding="utf-8")
        return json.loads(raw, object_hook=task_hook)

    def _save(self, tasks: list[Task]) -> None:
        payload = json.dumps([asdict(task) for task in tasks],
                             default=json_default, indent=2)
        self.path.write_text(payload, encoding="utf-8")

    # -- commands -------------------------------------------------------
    def add(self, title: str, priority: Priority = Priority.MEDIUM,
            due: date | None = None) -> Task:
        tasks = self._load()
        next_id = max((task.id for task in tasks), default=0) + 1
        task = Task(id=next_id, title=title, priority=priority, due=due)
        tasks.append(task)
        self._save(tasks)
        return task

    def get(self, task_id: int) -> Task:
        for task in self._load():
            if task.id == task_id:
                return task
        raise TaskNotFoundError(task_id)

    def list_tasks(self, *, status: Status | None = None,
                   priority: Priority | None = None) -> list[Task]:
        return [task for task in self._load()        # (**3.3**) filter
                if (status is None or task.status is status)
                and (priority is None or task.priority is priority)]

    def complete(self, task_id: int) -> Task:
        tasks = self._load()
        for task in tasks:
            if task.id == task_id:
                task.status = Status.DONE
                self._save(tasks)
                return task
        raise TaskNotFoundError(task_id)

    def remove(self, task_id: int) -> Task:
        tasks = self._load()
        keep = [task for task in tasks if task.id != task_id]
        if len(keep) == len(tasks):
            raise TaskNotFoundError(task_id)
        self._save(keep)
        (removed,) = (task for task in tasks if task.id == task_id)
        return removed


print("TaskStore defined:", [m for m in vars(TaskStore) if not m.startswith("_")])

In [ ]:
store = TaskStore(WORK / "tasks.json")          # the path is OURS to choose

a = store.add("Ship the release", Priority.HIGH, due=date(2026, 8, 28))
b = store.add("Update the changelog")
c = store.add("Fix flaky login test", Priority.HIGH)
print(f"ids assigned in order: {a.id}, {b.id}, {c.id}")

print("\n-- what is actually on disk --")
print(store.path.read_text(encoding="utf-8"))

In [ ]:
# Filtered listing (a comprehension under the hood, **3.3**):
high = store.list_tasks(priority=Priority.HIGH)
print("high priority :", [task.title for task in high])

# Complete one, remove one:
store.complete(b.id)
store.remove(c.id)
done = store.list_tasks(status=Status.DONE)
print("done          :", [task.title for task in done])

# ⚠️ THE persistence test: a brand-new store on the same path -- as a new
# process would be -- must see identical data, with real types intact.
reopened = TaskStore(store.path)
assert reopened.list_tasks() == store.list_tasks()
assert reopened.get(a.id).due == date(2026, 8, 28)      # a date, not a str
print("reopened store:", [(task.id, task.title) for task in reopened.list_tasks()])

### Stage 4 — a command dispatcher with `match`/`case`

A CLI command is a **list of strings** — exactly what `sys.argv[1:]` gives you. That shape is
what structural pattern matching (**3.4**) was built for: literal prefixes select the command,
capture patterns pull out the arguments, a **guard** rejects a non-numeric id, and the wildcard
turns everything else into `InvalidCommandError`.

Three deliberate choices:

- `run_command` **returns** the output string and **raises** on failure — it never prints.
  The thin `main()` wrapper in section 4 does the printing and the catching. That split is why
  the tests in section 5 can assert on behaviour directly.
- ⚠️ No `input()` loop. Feeding scripted argv-lists is how real CLIs are driven (and tested);
  an interactive loop would make this notebook non-runnable and the code untestable.
- Only `priority` gets the friendly-error wrapper (`parse_priority`). A bad `--status` value
  or a malformed date still surfaces as a raw traceback — deliberate scope for this build;
  wrapping them the same way is extension exercise 6.

In [ ]:
USAGE = """taskr -- a tiny task tracker
  add TITLE [PRIORITY] [DUE]   create a task (priority: low|medium|high)
  list [--priority P | --status S]
  done ID                      mark a task completed
  rm ID                        delete a task"""


def format_task(task: Task) -> str:
    """One aligned row per task, via format specs in the f-string."""
    box = "x" if task.status is Status.DONE else " "
    due = task.due.isoformat() if task.due else "-"
    return f"[{box}] #{task.id:<3} {task.priority:<6} due {due:<10} {task.title}"


def render(tasks: list[Task]) -> str:
    if not tasks:
        return "(no tasks)"
    return "\n".join(format_task(task) for task in tasks)


def parse_priority(raw: str) -> Priority:
    try:
        return Priority(raw)
    except ValueError as exc:            # (**6.2**) chain -- don't discard the cause
        options = ", ".join(Priority)
        raise InvalidCommandError(
            f"unknown priority {raw!r} (choose from: {options})") from exc


def run_command(store: TaskStore, argv: list[str]) -> str:
    """Execute ONE argv-style command and return the text to print (**3.4**)."""
    match argv:
        case ["add", title]:
            task = store.add(title)
            return f"added #{task.id}: {task.title}"
        case ["add", title, priority]:
            task = store.add(title, parse_priority(priority))
            return f"added #{task.id}: {task.title} ({task.priority})"
        case ["add", title, priority, due]:
            task = store.add(title, parse_priority(priority),
                             due=date.fromisoformat(due))
            return f"added #{task.id}: {task.title} ({task.priority}, due {task.due})"
        case ["list"]:
            return render(store.list_tasks())
        case ["list", "--priority", raw]:
            return render(store.list_tasks(priority=parse_priority(raw)))
        case ["list", "--status", raw]:
            return render(store.list_tasks(status=Status(raw)))
        case ["done", task_id] if task_id.isdigit():     # (**3.4**) guard
            task = store.complete(int(task_id))
            return f"completed #{task.id}: {task.title}"
        case ["rm", task_id] if task_id.isdigit():
            task = store.remove(int(task_id))
            return f"removed #{task.id}: {task.title}"
        case []:
            return USAGE
        case _:
            raise InvalidCommandError(f"could not understand: {' '.join(argv)!r}")


demo = TaskStore(WORK / "dispatch.json")
for argv in (["add", "Write the design doc", "high"],
             ["add", "Book the standup room"],
             ["list"]):
    print(f"$ taskr {shlex.join(argv)}")     # shlex.join quotes for the banner
    print(run_command(demo, argv), end="\n\n")

### Stage 5 — the real-world variant: `argparse`

The `match`/`case` dispatcher is honest and readable — and for a four-command tool it is
genuinely enough. But the standard-library answer, and what you will meet in real codebases,
is **`argparse` with subparsers**: it generates `--help`, validates choices, converts types,
and reports errors in the conventional Unix format.

Note how much of our earlier design slots straight in:

- `type=Priority` works because `Priority("high")` already parses strings (**5.3**);
- `choices=Priority` works because an enum is iterable — and shows as `{low,medium,high}` in
  the subcommand's help because each `StrEnum` member formats as its value;
- `type=date.fromisoformat` reuses the exact converter from stage 2.

⚠️ We call `parse_args` on **scripted lists** (its optional argument) and show the help via
`format_help()` — because with no arguments `parse_args` reads the real `sys.argv`, and
`--help` (or any parse *error*) calls `sys.exit()`, which would stop the notebook kernel.
(In a shipped module you would wrap this construction in a `build_parser()` function; it is
flat here so we can print the subcommand help too.)

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="taskr", description="A tiny task tracker.")
sub = parser.add_subparsers(dest="command", required=True)

add = sub.add_parser("add", help="create a task")
add.add_argument("title")
add.add_argument("--priority", type=Priority, choices=Priority,
                 default=Priority.MEDIUM)
add.add_argument("--due", type=date.fromisoformat, metavar="YYYY-MM-DD")

lst = sub.add_parser("list", help="list tasks")
lst.add_argument("--priority", type=Priority, choices=Priority)

done = sub.add_parser("done", help="mark a task completed")
done.add_argument("id", type=int)

print(parser.format_help())              # what `taskr --help` would print
print(add.format_help())                 # what `taskr add --help` would print

args = parser.parse_args(["add", "Read the argparse docs",
                          "--priority", "low", "--due", "2026-09-15"])
print("parsed  :", args)                 # types already converted

arg_store = TaskStore(WORK / "argparse.json")
match args.command:                      # same dispatch idea, tiny surface
    case "add":
        task = arg_store.add(args.title, args.priority, due=args.due)
        print(f"executed: added #{task.id}: {task.title} "
              f"({task.priority}, due {task.due})")

## 4. A real session

The last missing piece is `main()` — four lines that decide the program's *personality*:

- a `TaskrError` becomes a one-line `taskr: error: ...` message (**6.2**'s single-base payoff);
- anything else — a real bug — still tracebacks, because hiding bugs behind a friendly
  message is how they survive to production.

Then a scripted session, exactly as argv would arrive from a shell: three `add`s, a filtered
`list`, a `done`, a `done` for an id that **does not exist**, and a final `list`.

In [ ]:
def main(store: TaskStore, argv: list[str]) -> None:
    """Run one command; expected failures become one friendly line."""
    try:
        print(run_command(store, argv))
    except TaskrError as err:            # ONE clause covers the whole family
        print(f"taskr: error: {err}")


session = [
    ["add", "Ship v2.1 release notes", "high", "2026-08-28"],
    ["add", "Refactor the auth tests"],
    ["add", "Water the office plants", "low", "2026-09-01"],
    ["list", "--priority", "high"],
    ["done", "2"],
    ["done", "99"],                      # <- no such task
    ["list"],
]

cli = TaskStore(WORK / "session.json")
for argv in session:
    print(f"$ taskr {shlex.join(argv)}")
    main(cli, argv)
    print()

## 5. Tests — real `pytest`, real output

The pattern from **15.3**: build a **temporary project directory**, write the code and the
tests into it, run `pytest` in a subprocess, and read genuine output. Nothing lands in the
repo, and what you see is exactly what a terminal would show.

This is also where the dependency-injection decision (**5.4**) pays out: the `store` fixture
(**15.4**) hands every test a `TaskStore` pointed at pytest's `tmp_path` — each test gets a
fresh, isolated store, and **no mocking or patching** (**15.5**) is needed anywhere, because
the only dependency was injectable from day one.

What we test:

| File | Proves |
|---|---|
| `test_store.py` | ids are sequential; types survive the round trip; data survives a *new* store; `complete` moves status; a missing id raises with the id attached |
| `test_dispatch.py` | `add` → `list` round trip through the CLI layer; unknown commands raise; a bad priority **chains** the original `ValueError` (**6.2**) |

In [ ]:
def make_project(files: dict[str, str], name: str = "proj") -> Path:
    """Create a temp project from {relative path: source}; return its path."""
    project = Path(tempfile.mkdtemp(prefix=f"{name}_", dir=WORK))
    for relpath, source in files.items():
        target = project / relpath
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    return project


def run_pytest(project: Path, *args: str) -> str:
    """Run pytest inside `project` and return its combined output (**15.3**)."""
    cmd = [sys.executable, "-m", "pytest", "--no-header",
           "-p", "no:cacheprovider", *args]
    done = subprocess.run(cmd, cwd=project, capture_output=True, text=True,
                          encoding="utf-8", errors="replace", timeout=300)
    banner = f"$ pytest {' '.join(args)}".rstrip()
    return (f"{banner}\n{'-' * 70}\n"
            f"{(done.stdout + done.stderr).rstrip()}\n"
            f"{'-' * 70}\nexit code: {done.returncode}")


version = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                         capture_output=True, text=True)
print("pytest :", version.stdout.strip())

In [ ]:
# The code from stages 1-4, assembled into ONE importable module -- the file
# you would ship (section 6's packaging exercise turns it into a real install).
TASKR_SRC = """
# taskr.py -- a tiny task tracker: model, JSON store, dispatcher.
from __future__ import annotations

import json
from dataclasses import asdict, dataclass
from datetime import date
from enum import StrEnum
from pathlib import Path


class Priority(StrEnum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


class Status(StrEnum):
    OPEN = "open"
    DONE = "done"


@dataclass
class Task:
    id: int
    title: str
    priority: Priority = Priority.MEDIUM
    status: Status = Status.OPEN
    due: date | None = None


class TaskrError(Exception):
    "Base for every failure taskr raises deliberately."


class TaskNotFoundError(TaskrError):
    def __init__(self, task_id: int) -> None:
        super().__init__(f"no task with id {task_id}")
        self.task_id = task_id


class InvalidCommandError(TaskrError):
    "The command line could not be understood."


TASK_FIELDS = {"id", "title", "priority", "status", "due"}


def json_default(obj: object) -> str:
    if isinstance(obj, date):
        return obj.isoformat()
    raise TypeError(f"not JSON-serialisable: {obj!r}")


def task_hook(raw: dict) -> Task | dict:
    if set(raw) != TASK_FIELDS:
        return raw
    return Task(
        id=raw["id"],
        title=raw["title"],
        priority=Priority(raw["priority"]),
        status=Status(raw["status"]),
        due=date.fromisoformat(raw["due"]) if raw["due"] else None,
    )


class TaskStore:
    "All persistence, behind one injected Path."

    def __init__(self, path: Path) -> None:
        self.path = path

    def _load(self) -> list[Task]:
        if not self.path.exists():
            return []
        raw = self.path.read_text(encoding="utf-8")
        return json.loads(raw, object_hook=task_hook)

    def _save(self, tasks: list[Task]) -> None:
        payload = json.dumps([asdict(task) for task in tasks],
                             default=json_default, indent=2)
        self.path.write_text(payload, encoding="utf-8")

    def add(self, title: str, priority: Priority = Priority.MEDIUM,
            due: date | None = None) -> Task:
        tasks = self._load()
        next_id = max((task.id for task in tasks), default=0) + 1
        task = Task(id=next_id, title=title, priority=priority, due=due)
        tasks.append(task)
        self._save(tasks)
        return task

    def get(self, task_id: int) -> Task:
        for task in self._load():
            if task.id == task_id:
                return task
        raise TaskNotFoundError(task_id)

    def list_tasks(self, *, status: Status | None = None,
                   priority: Priority | None = None) -> list[Task]:
        return [task for task in self._load()
                if (status is None or task.status is status)
                and (priority is None or task.priority is priority)]

    def complete(self, task_id: int) -> Task:
        tasks = self._load()
        for task in tasks:
            if task.id == task_id:
                task.status = Status.DONE
                self._save(tasks)
                return task
        raise TaskNotFoundError(task_id)

    def remove(self, task_id: int) -> Task:
        tasks = self._load()
        keep = [task for task in tasks if task.id != task_id]
        if len(keep) == len(tasks):
            raise TaskNotFoundError(task_id)
        self._save(keep)
        (removed,) = (task for task in tasks if task.id == task_id)
        return removed


def format_task(task: Task) -> str:
    box = "x" if task.status is Status.DONE else " "
    due = task.due.isoformat() if task.due else "-"
    return f"[{box}] #{task.id:<3} {task.priority:<6} due {due:<10} {task.title}"


def render(tasks: list[Task]) -> str:
    if not tasks:
        return "(no tasks)"
    return "\\n".join(format_task(task) for task in tasks)


def parse_priority(raw: str) -> Priority:
    try:
        return Priority(raw)
    except ValueError as exc:
        options = ", ".join(Priority)
        raise InvalidCommandError(
            f"unknown priority {raw!r} (choose from: {options})") from exc


USAGE = (
    "taskr -- a tiny task tracker\n"
    "  add TITLE [PRIORITY] [DUE]   create a task (priority: low|medium|high)\n"
    "  list [--priority P | --status S]\n"
    "  done ID                      mark a task completed\n"
    "  rm ID                        delete a task"
)


def run_command(store: TaskStore, argv: list[str]) -> str:
    match argv:
        case ["add", title]:
            task = store.add(title)
            return f"added #{task.id}: {task.title}"
        case ["add", title, priority]:
            task = store.add(title, parse_priority(priority))
            return f"added #{task.id}: {task.title} ({task.priority})"
        case ["add", title, priority, due]:
            task = store.add(title, parse_priority(priority),
                             due=date.fromisoformat(due))
            return f"added #{task.id}: {task.title} ({task.priority}, due {task.due})"
        case ["list"]:
            return render(store.list_tasks())
        case ["list", "--priority", raw]:
            return render(store.list_tasks(priority=parse_priority(raw)))
        case ["list", "--status", raw]:
            return render(store.list_tasks(status=Status(raw)))
        case ["done", task_id] if task_id.isdigit():
            task = store.complete(int(task_id))
            return f"completed #{task.id}: {task.title}"
        case ["rm", task_id] if task_id.isdigit():
            task = store.remove(int(task_id))
            return f"removed #{task.id}: {task.title}"
        case []:
            return USAGE
        case _:
            raise InvalidCommandError(f"could not understand: {' '.join(argv)!r}")
"""

print(f"taskr.py: {len(TASKR_SRC.splitlines()) - 1} lines")

In [ ]:
# conftest.py: the fixture every test file can request by NAME (**15.4**).
# tmp_path is pytest's own per-test temp directory -- injection meets isolation.
CONFTEST = """
import pytest

import taskr


@pytest.fixture
def store(tmp_path):
    "A TaskStore writing inside this test's private tmp_path."
    return taskr.TaskStore(tmp_path / "tasks.json")
"""

TEST_STORE = """
from datetime import date

import pytest

import taskr


def test_add_assigns_sequential_ids(store):
    first = store.add("one")
    second = store.add("two")
    assert (first.id, second.id) == (1, 2)


def test_round_trip_preserves_types(store):
    store.add("typed", taskr.Priority.HIGH, due=date(2026, 9, 1))
    (loaded,) = store.list_tasks()
    assert loaded.priority is taskr.Priority.HIGH
    assert loaded.due == date(2026, 9, 1)       # a real date came back


def test_persistence_survives_a_new_store(store):
    store.add("durable")
    reopened = taskr.TaskStore(store.path)      # a "new process"
    assert [task.title for task in reopened.list_tasks()] == ["durable"]


def test_complete_moves_task_to_done(store):
    task = store.add("finish me")
    store.complete(task.id)
    assert store.get(task.id).status is taskr.Status.DONE


def test_missing_id_raises_with_the_id_attached(store):
    with pytest.raises(taskr.TaskNotFoundError) as excinfo:
        store.complete(99)
    assert excinfo.value.task_id == 99
"""

print("conftest.py + test_store.py ready")

In [ ]:
TEST_DISPATCH = """
import pytest

import taskr


def test_add_then_list_shows_the_task(store):
    taskr.run_command(store, ["add", "Write tests", "high"])
    out = taskr.run_command(store, ["list", "--priority", "high"])
    assert "Write tests" in out


def test_unknown_command_is_a_taskr_error(store):
    with pytest.raises(taskr.InvalidCommandError):
        taskr.run_command(store, ["teleport", "somewhere"])


def test_bad_priority_chains_the_original_error(store):
    with pytest.raises(taskr.InvalidCommandError) as excinfo:
        taskr.run_command(store, ["add", "x", "urgent"])
    assert isinstance(excinfo.value.__cause__, ValueError)   # (**6.2**)
"""

project = make_project({
    "taskr.py": TASKR_SRC,
    "conftest.py": CONFTEST,
    "test_store.py": TEST_STORE,
    "test_dispatch.py": TEST_DISPATCH,
}, name="taskr_tests")

print(run_pytest(project, "-v"))

---

## Common Mistakes & Pitfalls

1. ⚠️ **Storing dates and enums as whatever `str()` gives you** instead of deciding the wire
   format. `str(date)` happens to be ISO — but code that *happens* to work is not designed to
   work. `default=`/`object_hook` (**8.3**) make the conversion a stated contract, and the
   round-trip `assert` in stage 2 makes it enforced.
2. ⚠️ **Catching `Exception` in `main()`.** It feels safe, but it converts real bugs into a
   polite one-liner — a `TypeError` from broken code deserves a traceback. Catch **your**
   base class, `TaskrError`, and nothing wider (**6.2**).
3. **Hard-coding the storage path** (`Path.home() / ".taskr.json"`) inside `TaskStore`. Now
   tests write into your real home directory, or need to patch. One constructor parameter
   (**5.4**) removed the entire problem — notice section 5 contains zero mocks.
4. **Printing from the store or the dispatcher.** Once logic prints, tests must capture stdout
   and reuse means "reformat with regexes". Return strings, raise exceptions; only `main()`
   prints.
5. ⚠️ **`raise InvalidCommandError(...)` without `from exc`** when re-raising a parse failure.
   The chained `ValueError` is the evidence in the traceback — and
   `test_bad_priority_chains_the_original_error` would fail without it (**6.2**).
6. **Building the CLI around `input()` loops.** It cannot be scripted, cannot be tested, and
   is not how shells invoke programs. Argv in, text out.
7. **`parser.parse_args()` with no argument in a notebook or test** — it reads the real
   `sys.argv` (Jupyter's own flags), and any error calls `sys.exit()`. Pass explicit lists.
8. **Comparing enum members with `==` against the wrong thing.** `Task.status == "done"`
   works with `StrEnum` — but `status is Status.DONE` states the intent and cannot silently
   compare a typo'd `"Done"` as merely unequal-but-fine.

## Best Practices

- **Design the errors before the features.** One project-wide base exception; specifics
  carry structured attributes (`task_id`), not just message text (**6.2**).
- **Make illegal states unrepresentable early**: `Priority("urgent")` raises at the boundary,
  so no `"urgent"` ever reaches the file (**5.3**).
- **One class owns the filesystem**, and its path is injected (**5.4**). Everything above it is
  pure "values in, values out" — which is why testing it needed nothing but `tmp_path`.
- **Assert the round trip** the moment you write a serialiser: `loads(dumps(x)) == x` with a
  dataclass `__eq__` is one line and catches every lossy-conversion bug forever (**8.3**).
- **Separate deciding from printing.** `run_command` returns text; `main` prints it. The
  boundary is exactly where the tests attach.
- **Let the pattern matching carry the grammar** (**3.4**): each `case` line *is* the usage
  line — compare `USAGE` with the `match` arms.
- **Graduate to `argparse` when flags arrive.** Positional grammars suit `match`/`case`;
  `--flags`, defaults, help text and choices are `argparse`'s whole job.

## Extension exercises

Each of these is a genuine, sized-for-an-evening improvement:

1. **Sort by due date.** Add `list --sort due` — `sorted(tasks, key=...)` with tasks lacking a
   due date last (hint: a key returning a tuple; `(task.due is None, task.due)` sorts `None`s
   to the end — check why that works before trusting it).
2. **Tabular output.** Replace `format_task` with a real table: compute the widest title, then
   use dynamic format specs (`f"{title:<{width}}"`) for aligned columns and a header row.
3. **A `search` command.** `taskr search REVIEW` → case-insensitive substring match over
   titles, as one comprehension in `TaskStore`; then a `case ["search", term]` arm and a test.
4. **Package it** (**17.2**). Write a `pyproject.toml` with
   `[project.scripts] taskr = "taskr:cli"`, add a `cli()` that does
   `main(TaskStore(Path.home() / ".taskr.json"), sys.argv[1:])`, and `pip install -e .` in a
   scratch venv — at which point `taskr add "..."` works in a real shell.
5. **Harden the store.** What happens today if `tasks.json` contains invalid JSON — which
   exception escapes, and is it a `TaskrError`? Decide what *should* happen, wrap it
   (`raise ... from exc`), and pin it with a test.
6. **Friendly errors everywhere.** Wrap `Status(raw)` and `date.fromisoformat(due)` the
   way `parse_priority` wraps `Priority` — chain the original error (**6.2**) — and add
   tests proving a raw traceback never reaches the user.

In [ ]:
# ---- Cleanup: remove every file this notebook created ----
shutil.rmtree(WORK, ignore_errors=True)
print(f"removed {WORK}")
print(f"still exists? {WORK.exists()}")

---

## Where this came from — and where next

| Ingredient | Notebook |
|---|---|
| `@dataclass`, `StrEnum` | **5.3** |
| injected dependencies | **5.4** |
| exception hierarchies & `raise ... from` | **6.2** |
| `pathlib`, standard library layout | **07** |
| JSON and its lossy conversions | **8.3** |
| `match`/`case` on sequences, guards | **3.4** |
| comprehensions as query language | **3.3** |
| type hints throughout | **4.5**, folder **16** |
| pytest, `tmp_path`, fixtures, no-mocks testing | **15.3–15.5** |

**Next steps in the curriculum:** folder **16** would let `mypy` check every signature here;
**17.2** packages `taskr` into an installable tool; and the other capstones in this folder
apply the same build-in-layers discipline to different domains.